# 3D N=64 Training on Colab (H100 80GB)
The experiment that failed on laptop after 14 hours. H100 should handle it.

In [ ]:
!nvidia-smi

In [ ]:
from google.colab import files
uploaded = files.upload()  # upload M34Project.zip

In [ ]:
!unzip -q M34Project.zip -d /content/project
%cd /content/project

In [ ]:
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install scipy numpy matplotlib -q

In [ ]:
import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Train 3D N=64 from N=32 curriculum weights
import torch
import numpy as np
import time
import json
from pathlib import Path
import sys
sys.path.insert(0, '.')

from src.data.poisson import assemble_poisson_3d
from src.models.unet import UNet
from src.training.losses import ConditionLoss
import torch.optim as optim

device = torch.device('cuda')
N = 64
epochs = 500
steps = 100
probes = 128
probe_batch = 128  # H100 80GB — go all in
lr = 1e-3
base_features = 32

print(f'3D N={N} ({N**3:,} DOFs), bf={base_features}, mode=conv')
model = UNet(base_features=base_features, levels=3, dim=3).to(device)
params = sum(p.numel() for p in model.parameters())
print(f'Model: {params:,} params')

# Load N=32 curriculum weights if available
n32_path = Path('results/curriculum/3d/N32/best.pt')
if n32_path.exists():
    model.load_state_dict(torch.load(n32_path, map_location=device, weights_only=True))
    print('Loaded N=32 curriculum weights (fine-tuning)')
    lr = 3e-4  # lower LR for fine-tuning
else:
    print('Training from scratch (no N=32 weights found)')

A = assemble_poisson_3d(N)
loss_fn = ConditionLoss(A, N, num_probes=probes, dim=3, mode='conv').to(device)
optimizer = optim.Adam(model.parameters(), lr=lr)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
scaler = torch.amp.GradScaler('cuda')

save_dir = Path('results/3d_n64_colab')
save_dir.mkdir(parents=True, exist_ok=True)

best_loss = float('inf')
loss_history = []
t_start = time.time()

for epoch in range(1, epochs + 1):
    model.train()
    epoch_loss = 0.0
    for step in range(steps):
        optimizer.zero_grad()
        loss = loss_fn(model, device, use_amp=True, probe_batch_size=probe_batch)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
    scheduler.step()
    avg_loss = epoch_loss / steps
    loss_history.append(avg_loss)
    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save(model.state_dict(), save_dir / 'best.pt')
        marker = ' *'
    else:
        marker = ''
    elapsed = time.time() - t_start
    print(f'Epoch {epoch:4d}/{epochs} | loss={avg_loss:.4f} | best={best_loss:.4f} | {elapsed/60:.0f}m{marker}')

torch.save(model.state_dict(), save_dir / 'final.pt')
np.save(save_dir / 'loss_history.npy', np.array(loss_history))
print(f'\nDone. Best loss: {best_loss:.4f}. Time: {(time.time()-t_start)/3600:.1f} hours')

In [ ]:
# Test the model
from src.solvers.cg import conjugate_gradient
from src.solvers.fcg import flexible_cg
from src.solvers.pcg import preconditioned_cg
from src.solvers.preconditioners import ic0_preconditioner
from src.evaluation.nn_preconditioner import make_nn_preconditioner
from src.data.generate import generate_source_term_3d
from src.data.poisson import get_grid_points_3d, assemble_rhs_3d

model.load_state_dict(torch.load('results/3d_n64_colab/best.pt', map_location=device, weights_only=True))
nn = make_nn_preconditioner(model, N, device=device, dim=3)
ic0 = ic0_preconditioner(A, grid_N=N, dim=3)

rng = np.random.default_rng(99)
cg_i, ic0_i, nn_i = [], [], []
for s in range(10):
    X, Y, Z = get_grid_points_3d(N)
    f = generate_source_term_3d(X, Y, Z, rng)
    b = assemble_rhs_3d(f, N)
    res_cg = conjugate_gradient(A, b, tol=1e-6)
    res_ic0 = preconditioned_cg(A, b, ic0, tol=1e-6)
    res_nn = flexible_cg(A, b, nn, tol=1e-6, max_iter=1000, m_max=20)
    cg_i.append(res_cg.iterations)
    ic0_i.append(res_ic0.iterations)
    nn_i.append(res_nn.iterations)
    print(f'Sample {s+1}/10: CG={res_cg.iterations}, IC(0)={res_ic0.iterations}, NN={res_nn.iterations}')

print(f'\n3D N={N} ({N**3:,} DOFs):')
print(f'CG:     {np.mean(cg_i):.1f} iters')
print(f'IC(0):  {np.mean(ic0_i):.1f} iters ({1-np.mean(ic0_i)/np.mean(cg_i):.1%})')
print(f'NN+FCG: {np.mean(nn_i):.1f} iters ({1-np.mean(nn_i)/np.mean(cg_i):.1%})')

In [ ]:
# Download models
from google.colab import files
files.download('results/3d_n64_colab/best.pt')
files.download('results/3d_n64_colab/final.pt')